<a href="https://colab.research.google.com/github/areslou/cpecog2-coral-bleaching/blob/main/notebooks/Group2_Coral_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coral bleaching demo - Run 5 R14

This notebook launches the coral bleaching classification app in Google Colab.

- Model: Run 5 R14, seed 42
- File: `results/run5/final_model_s42.pt`
- Bleaching cutoff: `0.285`
- The model is a screening aid; a trained reef monitor makes the final decision.

For a browser demo, use a GPU runtime and run every cell. Keep the final cell running while using the `gradio.live` link.

In [ ]:
!if [ -d cpecog2-coral-bleaching/.git ]; then cd cpecog2-coral-bleaching; else git clone https://github.com/areslou/cpecog2-coral-bleaching.git && cd cpecog2-coral-bleaching; fi

In [ ]:
!pip install -q ultralytics==8.4.211 pi-heif gradio==6.32.0 torch torchvision kagglehub pandas pillow numpy

In [ ]:
import os
import torch

os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
os.environ.setdefault("YOLO_AUTOINSTALL", "false")
os.environ.setdefault("YOLO_OFFLINE", "true")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("PyTorch:", torch.__version__)

## 1. Check the Run 5 model

The model should be downloaded as part of the repository. The model card specifies the architecture and class order.

In [ ]:
WEIGHTS = "results/run5/final_model_s42.pt"
THRESHOLD = 0.285

if not os.path.exists(WEIGHTS):
    raise FileNotFoundError(
        f"{WEIGHTS} was not found. Run this notebook from the repository with Run 5 weights, "
        "or update WEIGHTS to the downloaded model path."
    )

import json
card_path = "results/run5/final_model_s42.json"
if os.path.exists(card_path):
    print(json.dumps(json.load(open(card_path, encoding="utf-8")), indent=2))
print("Weights:", WEIGHTS)
print("Threshold:", THRESHOLD)
print("Size MB:", round(os.path.getsize(WEIGHTS) / 1e6, 2))

## 2. Optional: create demo photographs

This downloads the held-out test photographs once. Skip this cell if you will upload your own reef photographs.

In [ ]:
#!python get_demo_photos.py
#!python get_demo_photos.py --sites

## 3. Launch the demo app

The public link is created with Gradio `share=True`. Upload either:

- one reef photograph in the **Single photo** tab, or
- photographs and/or a `photo,site` CSV in the **Survey summary** tab.

In [ ]:
from app import build

ui = build(WEIGHTS, THRESHOLD)
ui.launch(server_name="0.0.0.0", server_port=7860, share=True)